# QIH-LLM Experiment — Period-Aware Fusion for Language Modeling

**Goal:** Show how QIH-PAT features (analytic QFT histograms per window) can boost small language models
on sequences that contain hidden periodic motifs.

## Baseline vs QIH: window classification task

## Features and models

## (Optional) Tiny LM with QIH Fusion (PyTorch)

In [ ]:
import numpy as np
from quantum_hybrid_system.tools_qih.qih_llm import qih_features_for_tokens, tokens_to_signal

rng = np.random.default_rng(123)

# ---- Synthetic corpus ----
V = 64  # vocab size
Tseq = 512
Nseq = 2000  # dataset size (keep modest)
motif = [7, 13, 29, 13]   # repeating motif

def make_seq(period=32, jitter=3, motif_prob=0.6):
    # base: random tokens
    x = rng.integers(0, V, size=Tseq).tolist()
    # inject repeated motif with probabilistic placement
    pos = rng.integers(0, 10)
    while pos + len(motif) < Tseq:
        if rng.random() < motif_prob:
            x[pos:pos+len(motif)] = motif
        pos += max(4, period + rng.integers(-jitter, jitter+1))
    return x

data = [make_seq() for _ in range(Nseq)]
labels = np.zeros(Nseq, dtype=int)  # we will create a window-level task later
print("Dataset ready:", len(data))

In [ ]:
W, S = 128, 64

def windows_tokens(tokens):
    for start in range(0, len(tokens)-W+1, S):
        yield tokens[start:start+W]

def window_has_motif(tokens, motif=(7,13,29,13)):
    for i in range(0, len(tokens)-len(motif)+1):
        if tokens[i:i+len(motif)] == list(motif):
            return True
    return False

Xw = []
Yw = []
for seq in data:
    for w in windows_tokens(seq):
        Xw.append(np.array(w, dtype=int))
        Yw.append(1 if window_has_motif(w.tolist()) else 0)

Xw = np.stack(Xw, axis=0)
Yw = np.array(Yw, dtype=int)
perm = rng.permutation(len(Xw))
Xw, Yw = Xw[perm], Yw[perm]

print("Windows:", Xw.shape, "pos rate:", float(Yw.mean()))

In [ ]:
def baseline_feats_tokens(Wx, V):
    m = Wx.mean(axis=1)
    s = Wx.std(axis=1)
    mx = Wx.max(axis=1)
    mn = Wx.min(axis=1)
    uniq = np.array([len(set(row.tolist())) for row in Wx])
    return np.vstack([m, s, mx, mn, uniq]).T

Xb = baseline_feats_tokens(Xw, V)

# QIH features per window
Hs = []
for i in range(len(Xw)):
    H, _ = qih_features_for_tokens(Xw[i], win=W, stride=W, bins=64)
    Hs.append(H[0])
H = np.stack(Hs, axis=0)

print("Feature shapes:", Xb.shape, H.shape)

In [ ]:
try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score
    split = int(0.8 * len(Xw))
    Xb_tr, Xb_te = Xb[:split], Xb[split:]
    H_tr, H_te = H[:split], H[split:]
    y_tr, y_te = Yw[:split], Yw[split:]

    base = LogisticRegression(max_iter=200).fit(Xb_tr, y_tr)
    qih  = LogisticRegression(max_iter=200).fit(np.hstack([Xb_tr, H_tr]), y_tr)

    pb = base.predict_proba(Xb_te)[:,1]
    pq = qih.predict_proba(np.hstack([Xb_te, H_te]))[:,1]

    auc_base = roc_auc_score(y_te, pb)
    auc_qih = roc_auc_score(y_te, pqp) if 'pqp' in locals() else roc_auc_score(y_te, pq)
    print({"auc_base": float(auc_base), "auc_qih": float(auc_qih)})
except Exception as e:
    print("sklearn not available:", e)

In [ ]:
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    from quantum_hybrid_system.tools_qih.qih_llm import QIHFusionEmbedding

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Build small dataset for next-token prediction
    Tclip = 256
    tok_arrays = [np.array(seq[:Tclip], dtype=int) for seq in data[:1024]]
    Xtok = np.stack([t[:-1] for t in tok_arrays], axis=0)  # [N, T-1]
    Ytok = np.stack([t[1:]  for t in tok_arrays], axis=0)  # next token

    ds = TensorDataset(torch.tensor(Xtok, dtype=torch.long),
                       torch.tensor(Ytok, dtype=torch.long))
    dl = DataLoader(ds, batch_size=32, shuffle=True)

    class TinyLM(nn.Module):
        def __init__(self, vocab, d=128):
            super().__init__()
            self.emb = nn.Embedding(vocab, d)
            self.proj = nn.Linear(d, vocab)
            self.rnn = nn.GRU(d, d, batch_first=True)
        def forward(self, x):
            e = self.emb(x)
            y, _ = self.rnn(e)
            return self.proj(y)

    class TinyLMQIH(nn.Module):
        def __init__(self, vocab, d=128, win=64, stride=64):
            super().__init__()
            self.emb = nn.Embedding(vocab, d)
            self.qih = QIHFusionEmbedding(win=win, stride=stride, bins=64, embed_dim=d)
            self.proj = nn.Linear(d, vocab)
            self.rnn = nn.GRU(d, d, batch_first=True)
        def forward(self, x):
            e = self.emb(x)
            e = self.qih(x, e)    # fuse QIH projection per window
            y, _ = self.rnn(e)
            return self.proj(y)

    def train_model(model, steps=100):
        model.to(device)
        opt = torch.optim.Adam(model.parameters(), lr=3e-3)
        lossf = nn.CrossEntropyLoss()
        it = iter(dl)
        model.train()
        losses = []
        for step in range(steps):
            try:
                xb, yb = next(it)
            except StopIteration:
                it = iter(dl)
                xb, yb = next(it)
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)  # [B, T, V]
            loss = lossf(logits.reshape(-1, logits.size(-1)), yb.reshape(-1))
            opt.zero_grad()
            loss.backward()
            opt.step()
            if step % 25 == 0:
                losses.append(float(loss.item()))
        return losses

    m_base = TinyLM(V)
    m_qih  = TinyLMQIH(V)

    lb = train_model(m_base, steps=100)
    lq = train_model(m_qih,  steps=100)

    print({"loss_trace_base": lb, "loss_trace_qih": lq})
except Exception as e:
    print("PyTorch path skipped:", e)